# ML Study Tracker 4: Practical ML Use Cases

This notebook focuses on the end-to-end practical application of machine learning workflows for core business and public dataset use cases: Linear Regression for House Prices, Logistic Regression for Customer Churn, and Machine Learning-based Time Series Forecasting.

## Table of Contents
- [Common Imports & Configuration](#Common-Imports-&-Configuration)
- [1. Linear Regression for House Price Prediction](#1.-Linear-Regression-for-House-Price-Prediction)
- [2. Logistic Regression for Customer Churn Prediction](#2.-Logistic-Regression-for-Customer-Churn-Prediction)
- [3. Time Series Forecasting for Sales/Stock Prediction](#3.-Time-Series-Forecasting-for-Sales/Stock-Prediction)
- [Mini Project Tracker](#Mini-Project-Tracker)
- [Experiment Log](#Experiment-Log)

In [ ]:
# Common Imports & Configuration
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Set random seed for reproducibility across all use cases
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Linear Regression for House Price Prediction
**Dataset Target:** California Housing Dataset (Popular Public Benchmark)

### Study Checklist
- [ ] Data Exploration & Target Skew Check
- [ ] Feature Scaling
- [ ] Model Training (OLS vs. Regularized Ridge/Lasso)
- [ ] Evaluation Metrics (RMSE, MAE, R^2)
- [ ] Residual Analysis (Homoscedasticity check)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# Practical Implementation: House Price Prediction
from sklearn.linear_model import Ridge
from sklearn.datasets import fetch_california_housing

# 1. Fetch Popular Public Data Source
housing = fetch_california_housing(as_frame=True)
df = housing.frame

# 2. Setup Features and Target
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

# 4. Pipeline Construction (Scale -> Regularized Regression)
house_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

# 5. Fit & Evaluate
house_pipeline.fit(X_train, y_train)
y_pred = house_pipeline.predict(X_test)

print("--- House Price Prediction Metrics ---")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"R^2:  {r2_score(y_test, y_pred):.4f}")

# 6. Residual Verification
residuals = y_test - y_pred
print(f"Mean of Residuals (Want ~0): {residuals.mean():.4f}")

## 2. Logistic Regression for Customer Churn Prediction
**Dataset Target:** Imbalanced Customer Behavior/Churn Analysis

### Study Checklist
- [ ] Imbalance Check & Evaluation Metric Selection (Avoid raw Accuracy)
- [ ] Dummy Encoding / One-Hot Encoding for Categories
- [ ] Addressing Imbalance (Class weights vs. Resampling)
- [ ] Extracting Decision Probabilities & Threshold Tuning
- [ ] Coefficient Odds-Ratio Interpretation

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# Practical Implementation: Customer Churn Prediction
from sklearn.linear_model import LogisticRegression

# 1. Create a Synthetic Messy Churn Dataset to Simulate Production Data
n_samples = 1000
raw_data = pd.DataFrame({
    'Age': np.random.randint(18, 70, n_samples),
    'Tenure_Months': np.random.randint(0, 72, n_samples),
    'Contract_Type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples),
    'Monthly_Charges': np.random.uniform(20, 120, n_samples)
})

# Construct ground truth dependent on categorical and numeric patterns
churn_logit = (-2.0 
               + (raw_data['Contract_Type'] == 'Month-to-month') * 1.8 
               + raw_data['Monthly_Charges'] * 0.01 
               - raw_data['Tenure_Months'] * 0.03)
churn_prob = 1 / (1 + np.exp(-churn_logit))
raw_data['Churn'] = (np.random.rand(n_samples) < churn_prob).astype(int)

# 2. Identify Features and Target
numeric_features = ['Age', 'Tenure_Months', 'Monthly_Charges']
categorical_features = ['Contract_Type']

X = raw_data.drop(columns='Churn')
y = raw_data['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

# 3. Preprocessing Transformer Block
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first'), categorical_features)
])

# 4. Balanced Logistic Regression Pipeline
churn_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', random_state=RANDOM_SEED))
])

# 5. Fit, Predict Proba, and Evaluate via F1-Score
churn_pipeline.fit(X_train, y_train)
y_pred_proba = churn_pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print("--- Customer Churn Prediction Metrics ---")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")

## 3. Time Series Forecasting for Sales/Stock Prediction
**Dataset Target:** Supermarket Sales / Historical Demand Sequences

### Study Checklist
- [ ] Structural Decomposition (Trend, Seasonality, Residual Noise)
- [ ] Designing Lag Features without Future Data Leakage
- [ ] Adding Rolling Windows (Mean, Std)
- [ ] Executing Chronological Train/Test Time Split
- [ ] Autoregressive ML Evaluation Benchmark

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# Practical Implementation: Time Series Forecasting
from sklearn.linear_model import LinearRegression

# 1. Generate Synthetic Sequential Sales Time Series
t = np.arange(120)
trend = 0.3 * t
seasonality = 10 * np.sin(2 * np.pi * t / 7)  # Weekly cycle
noise = np.random.normal(0, 1.5, 120)
sales_series = pd.DataFrame({'Sales': 100 + trend + seasonality + noise})

# 2. Engineer Lag and Rolling Statistics Features
sales_series['Lag_1'] = sales_series['Sales'].shift(1)
sales_series['Lag_7'] = sales_series['Sales'].shift(7)
sales_series['Rolling_Mean_3'] = sales_series['Sales'].shift(1).rolling(window=3).mean()

# Drop rows with NaN values caused by shifts/rolling windows
sales_series.dropna(inplace=True)

# 3. Chronological Time-based Train/Test Split
X_ts = sales_series[['Lag_1', 'Lag_7', 'Rolling_Mean_3']]
y_ts = sales_series['Sales']

# Retain the final 14 steps cleanly for validation (No random shuffling!)
split_idx = len(sales_series) - 14
X_train_ts, X_test_ts = X_ts.iloc[:split_idx], X_ts.iloc[split_idx:]
y_train_ts, y_test_ts = y_ts.iloc[:split_idx], y_ts.iloc[split_idx:]

# 4. Fit Linear Autoregressive Forecasting Model
ts_model = LinearRegression().fit(X_train_ts, y_train_ts)
ts_preds = ts_model.predict(X_test_ts)

print("--- Sequential Time Series Metrics ---")
print(f"Forecast MAE:  {mean_absolute_error(y_test_ts, ts_preds):.4f}")
print(f"Forecast RMSE: {np.sqrt(mean_squared_error(y_test_ts, ts_preds)):.4f}")

## Mini Project Tracker

| Project | Topic | Dataset Benchmark | Model Baseline | Primary Metric | Status | Notes |
|---|---|---|---|---|---|---|
| **House Price Prediction** | Regression | California Housing | `Pipeline(StandardScaler, Ridge)` | RMSE / R^2 | Not Started | Focus on linear assumptions & coefficients |
| **Customer Churn Prediction** | Classification | Synthetic Tabular | `Pipeline(ColumnTransformer, LogisticRegression)` | F1-Score | Not Started | Map metrics and address target asymmetry |
| **Stock / Sales Forecasting** | Time Series | Synthetic Sequence | `LinearRegression(Lag + Rolling features)` | MAE / RMSE | Not Started | Enforce chronological split parameters |

## Experiment Log

| Date | Problem Context | Dataset Input | Model Configuration | Preprocessing Setup | Metric Tracked | Result | Next Progressive Steps |
|---|---|---|---|---|---|---|---| 
| YYYY-MM-DD | House Price Prediction | California Housing | Ridge (alpha=1.0) | Standard Scaled (All features) | R^2 Score | | Tune alpha parameters via GridSearchCV |